# Module 11: Time Series Visualization with Matplotlib
## Analyzing 42 Years of CHIRPS Rainfall (1981-2022)

This module covers time series visualization techniques: date formatting, rolling averages, seasonal decomposition, trend analysis, year-over-year comparisons, and climate driver analysis using CHIRPS rainfall data.

## Setup

Load required libraries and prepare time series data.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from scipy import stats
from scipy.signal import savgol_filter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
print('Libraries loaded successfully!')

In [ ]:
DATA_DIR = '../datasets'
CHIRPS_PATH = '../chirps_1981_2022.nc'

ds = xr.open_dataset(CHIRPS_PATH, engine='netcdf4')
point_df = pd.read_csv(f'{DATA_DIR}/addis_ababa_rainfall.csv', parse_dates=['time'])
regional = pd.read_csv(f'{DATA_DIR}/ethiopia_regional_rainfall.csv', parse_dates=['time'])

eth_mean = ds.precip.sel(latitude=slice(15, 3), longitude=slice(33, 48)).mean(dim=['latitude', 'longitude'])
eth_ts = eth_mean.to_dataframe().reset_index()

print(f'Point data: {point_df.shape}')
print(f'Regional data: {regional.shape}')
print(f'Ethiopia mean TS: {eth_ts.shape}')
print(f'Time range: {point_df.time.min()} to {point_df.time.max()}')

## 1. Date Formatting on Axes

Matplotlib's `mdates` module provides sophisticated date formatting. We'll plot the full 42-year time series with proper date formatting.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(point_df['time'], point_df['precip'],
        color='steelblue', linewidth=0.8, alpha=0.7, label='Monthly Rainfall')

ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_minor_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('CHIRPS Rainfall at Addis Ababa (1981-2022)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 2. Rolling Averages

Rolling (moving) averages smooth out short-term fluctuations to highlight longer-term trends. We'll apply 6-month and 12-month windows.

In [ ]:
point_df_sorted = point_df.sort_values('time').set_index('time')
point_df_sorted['roll_6mo'] = point_df_sorted['precip'].rolling(window=6, center=True).mean()
point_df_sorted['roll_12mo'] = point_df_sorted['precip'].rolling(window=12, center=True).mean()
point_df_sorted['roll_24mo'] = point_df_sorted['precip'].rolling(window=24, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(point_df_sorted.index, point_df_sorted['precip'],
        color='lightgray', linewidth=0.5, alpha=0.5, label='Monthly')
ax.plot(point_df_sorted.index, point_df_sorted['roll_6mo'],
        color='orange', linewidth=1.5, label='6-month MA')
ax.plot(point_df_sorted.index, point_df_sorted['roll_12mo'],
        color='red', linewidth=2.0, label='12-month MA')
ax.plot(point_df_sorted.index, point_df_sorted['roll_24mo'],
        color='darkblue', linewidth=2.5, label='24-month MA')

ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Rolling Averages of Monthly Rainfall at Addis Ababa', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 3. Seasonal Climatology and Anomalies

Decompose the time series into the average seasonal cycle and anomalies from that cycle.

In [ ]:
monthly_clim = point_df.groupby(point_df['time'].dt.month)['precip'].mean()
monthly_std = point_df.groupby(point_df['time'].dt.month)['precip'].std()

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(range(1, 13), monthly_clim.values, 'o-', color='darkgreen',
        linewidth=2.5, markersize=8, label='Mean')
ax.fill_between(range(1, 13),
                monthly_clim.values - monthly_std.values,
                monthly_clim.values + monthly_std.values,
                alpha=0.2, color='green', label='±1σ')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Average Seasonal Rainfall Cycle - Addis Ababa (1981-2022)', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
point_df['month'] = point_df['time'].dt.month
monthly_mean = point_df.groupby('month')['precip'].transform('mean')
point_df['anomaly'] = point_df['precip'] - monthly_mean

point_df_sorted = point_df.sort_values('time').set_index('time')

fig, ax = plt.subplots(figsize=(16, 6))

colors = np.where(point_df_sorted['anomaly'] >= 0, 'darkblue', 'firebrick')
ax.bar(point_df_sorted.index, point_df_sorted['anomaly'],
       color=colors, width=25, alpha=0.7, edgecolor='none')

ax.axhline(0, color='black', linewidth=0.8)

ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Rainfall Anomaly (mm/month)', fontsize=12)
ax.set_title('Monthly Rainfall Anomalies at Addis Ababa (1981-2022)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 4. Trend Analysis with Regression

Linear regression overlaid on annual rainfall totals reveals long-term trends.

In [ ]:
annual_eth = eth_ts.copy()
annual_eth['year'] = pd.to_datetime(annual_eth['time']).dt.year
annual_sum = annual_eth.groupby('year')['precip'].sum().reset_index()

x = annual_sum['year'].values
y = annual_sum['precip'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
trend_line = slope * x + intercept

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(annual_sum['year'], annual_sum['precip'],
       color='steelblue', alpha=0.6, edgecolor='navy', linewidth=0.5,
       label='Annual Rainfall')

ax.plot(x, trend_line, 'r-', linewidth=2.5, label=f'Trend: {slope:.1f} mm/yr (p={p_value:.3f})')

ax.fill_between(x, trend_line - std_err * x, trend_line + std_err * x,
                alpha=0.1, color='red', label='±1σ CI')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual Precipitation (mm/year)', fontsize=12)
ax.set_title(f'Annual Rainfall Trend over Ethiopia (1981-2022)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 5. Year-over-Year Comparison

Compare the monthly progression of rainfall across different years to see how individual years deviate from the normal cycle.

In [ ]:
point_df['year'] = point_df['time'].dt.year
point_df['month_num'] = point_df['time'].dt.month

selected_years = [1981, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2022]
yoy_data = point_df[point_df['year'].isin(selected_years)]

fig, ax = plt.subplots(figsize=(14, 8))

for year in selected_years:
    ydata = yoy_data[yoy_data['year'] == year]
    ax.plot(ydata['month_num'], ydata['precip'], 'o-', linewidth=1.5,
            markersize=5, label=str(year), alpha=0.8)

clim_mean = point_df.groupby('month_num')['precip'].mean()
ax.plot(range(1, 13), clim_mean.values, 'k-', linewidth=3,
        label='1981-2010 Mean', alpha=0.9)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Year-over-Year Monthly Rainfall Comparison - Addis Ababa', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(ncol=3, fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 6. Climate Drivers: El Niño/La Niña Years

Analyze rainfall patterns during El Niño and La Niña events. Known ENSO years are highlighted to identify teleconnection signals in Ethiopian rainfall.

In [ ]:
el_nino_years = [1982, 1987, 1991, 1997, 2002, 2009, 2015]
la_nina_years = [1984, 1988, 1995, 1998, 2000, 2007, 2010, 2020]
neutral_years = [y for y in range(1981, 2023)
                 if y not in el_nino_years and y not in la_nina_years]

annual_sum['enso'] = 'Neutral'
annual_sum.loc[annual_sum['year'].isin(el_nino_years), 'enso'] = 'El Niño'
annual_sum.loc[annual_sum['year'].isin(la_nina_years), 'enso'] = 'La Niña'

enso_colors = {'El Niño': '#e74c3c', 'La Niña': '#3498db', 'Neutral': '#95a5a6'}

fig, ax = plt.subplots(figsize=(15, 7))

for enso_type in ['El Niño', 'La Niña', 'Neutral']:
    mask = annual_sum['enso'] == enso_type
    ax.scatter(annual_sum.loc[mask, 'year'], annual_sum.loc[mask, 'precip'],
               s=80, c=enso_colors[enso_type], edgecolors='black', linewidths=0.5,
               label=enso_type, zorder=5)

ax.axhline(annual_sum['precip'].mean(), color='gray', linestyle='--',
           linewidth=1.5, alpha=0.7, label=f'Mean: {annual_sum["precip"].mean():.0f} mm')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual Precipitation (mm/year)', fontsize=12)
ax.set_title('Ethiopia Annual Rainfall by ENSO Phase (1981-2022)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
enso_groups = annual_sum.groupby('enso')['precip']
print('ENSO Rainfall Statistics:')
for name, group in enso_groups:
    print(f'  {name}: mean={group.mean():.0f} mm, std={group.std():.0f} mm, n={len(group)}')

el = annual_sum[annual_sum['enso'] == 'El Niño']['precip'].dropna().values
ln = annual_sum[annual_sum['enso'] == 'La Niña']['precip'].dropna().values
nt = annual_sum[annual_sum['enso'] == 'Neutral']['precip'].dropna().values

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for idx, (data, label, color) in enumerate(zip(
    [el, ln, nt], ['El Niño', 'La Niña', 'Neutral'],
    [enso_colors['El Niño'], enso_colors['La Niña'], enso_colors['Neutral']]):
    ax = axes[idx]
    ax.hist(data, bins=8, density=True, alpha=0.7,
            color=color, edgecolor='black', linewidth=0.8)
    if len(data) > 1:
        kde = stats.gaussian_kde(data)
        x_vals = np.linspace(data.min() - 50, data.max() + 50, 100)
        ax.plot(x_vals, kde(x_vals), 'k-', linewidth=2)
    ax.axvline(np.mean(data), color='darkred', linestyle='--', linewidth=1.5)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Annual Rainfall (mm)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Distribution of Annual Rainfall by ENSO Phase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Seasonal Decomposition

Decompose the time series into trend, seasonal, and residual components using a simple moving average approach.

In [ ]:
point_ts = point_df_sorted[['precip']].copy()

trend = point_ts['precip'].rolling(window=13, center=True, min_periods=6).mean()
detrended = point_ts['precip'] - trend

monthly_avg = detrended.groupby(detrended.index.month).mean()
seasonal = np.tile(monthly_avg.values, 42)[:len(detrended)]
residual = detrended - seasonal

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

axes[0].plot(point_ts.index, point_ts['precip'], color='steelblue', linewidth=0.8)
axes[0].set_ylabel('Original (mm)', fontsize=11)
axes[0].set_title('Time Series Decomposition of Addis Ababa Rainfall', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(point_ts.index, trend, color='red', linewidth=1.5)
axes[1].set_ylabel('Trend (mm)', fontsize=11)
axes[1].grid(True, alpha=0.3)

axes[2].plot(point_ts.index, seasonal, color='darkgreen', linewidth=1.2)
axes[2].set_ylabel('Seasonal (mm)', fontsize=11)
axes[2].grid(True, alpha=0.3)

axes[3].bar(point_ts.index, residual, color='gray', width=25, alpha=0.6)
axes[3].axhline(0, color='black', linewidth=0.5)
axes[3].set_ylabel('Residual (mm)', fontsize=11)
axes[3].grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

axes[-1].set_xlabel('Year', fontsize=12)

plt.tight_layout()
plt.show()

## 8. Monthly Rainfall Heatmap (Years × Months)

A 2D heatmap of years vs months reveals the full temporal structure at a glance.

In [ ]:
annual_cycle = point_df.pivot_table(
    index='year', columns='month_num', values='precip', aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(14, 10))

pcm = ax.pcolormesh(annual_cycle.columns, annual_cycle.index, annual_cycle.values,
                    cmap='YlGnBu', shading='auto', vmin=0, vmax=300)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Year', fontsize=12)
ax.set_title('Monthly Rainfall Heatmap - Addis Ababa (1981-2022)', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.invert_yaxis()

cbar = plt.colorbar(pcm, ax=ax, orientation='vertical', pad=0.01, aspect=30)
cbar.set_label('Precipitation (mm/month)', fontsize=11)

plt.tight_layout()
plt.show()

## Exercises

1. **Regional Time Series**: Plot monthly rainfall time series for 3 contrasting regions (Tigray, Oromia, Somali) on the same axes with proper date formatting.
2. **Rolling Correlation**: Compute and plot the 24-month rolling correlation between two regions.
3. **Trend per Season**: Compute and plot trends separately for each season (MAM, JJA, SON, DJF) over the 42-year period.
4. **ENSO Composite**: Create a bar chart comparing mean monthly rainfall during El Niño, La Niña, and Neutral years for the July-September period.
5. **Extreme Events**: Identify the top 10 wettest and driest months, and highlight them on the time series plot.

### Mini-Project: Ethiopian Rainfall Trend Analysis Report

Create a comprehensive time series analysis with:
- A 2×2 figure panel showing: original TS, 12-month rolling average, annual totals with trend line, and ENSO-colored scatter plot
- Seasonal decomposition plot for a specific region
- A heatmap of monthly rainfall anomalies (year × month)
- Statistical summary: trend slope, significance, and interpretation
- Discuss whether Ethiopian rainfall is increasing, decreasing, or stable